# Topomap plots — Flow & Load papers

**Usage** : set toggles in **Cell 2 (Config)**, run cells 1–5 to load data,  
then use **Cell 6 (Batch)** or **Cell 7 (Live)**.


In [ ]:
import sys
from datetime import datetime
from pathlib import Path
import sys

def find_repo_root(marker="core") -> Path:
    """Walk up from the current directory until a folder containing
    `marker` (e.g. core/) is found -- avoids hardcoding folder depth,
    which breaks every time notebooks get moved."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Could not locate repo root (no '{marker}/' found above {Path.cwd()})")

sys.path.insert(0, str(find_repo_root()))
from core import config_flow, config_load
from core.participants import get_participants
from core.utils import get_cluster_label, load_pkl, show_latest_pkls, tuple_to_string_fname, tuple_to_string_title, window_average_evoked, get_topomap_image

import matplotlib.pyplot as plt
import numpy as np
import mne
from mne import grand_average
from mne.defaults import DEFAULTS
from mne.viz import set_browser_backend


%matplotlib qt
set_browser_backend("qt")

plt.rcParams.update({
    "font.size":        12,
    "axes.titlesize":   13,
    "axes.labelsize":   12,
    "xtick.labelsize":  10,
    "ytick.labelsize":  10,
    "legend.fontsize":  10,
    "figure.titlesize": 16,
})


In [ ]:
# ==============================================================================
# Configuration — edit here, then re-run from Cell 3 downward
# ==============================================================================

PAPER:    str = "load"
PARADIGM: str = "passive"     # "active" | "passive" / "active-n35" 
MEASURE:  str = "diff"        # "raw" | "diff"
LOCK:     str = "stim"       # "stim" | "resp"

CSD:           bool = False
BASELINE_CORR: str  = "bc-200-100"

print(f"Paper: {PAPER.upper()} | Paradigm: {PARADIGM.upper()} | Measure: {MEASURE.upper()} | Lock: {LOCK.upper()}")

# Minimal derived — needed for show_latest_pkls
PARADIGM_REAL = PARADIGM.split("-")[0]
cohort        = PARADIGM.split("-", 1)[1] if "-" in PARADIGM else None
STUDY         = "Active" if PARADIGM_REAL == "active" else "Passive"

if sys.platform == "win32":
    root = Path(f"L:/EEG_Flow_data/Study_EEG_Flow_{STUDY}/derivatives_v2")
else:
    root = Path(f"/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_{STUDY}/derivatives_v2")
    
if PAPER == 'flow' and LOCK == 'resp':
    raise ValueError(
        "Flow response-locked analyses use CSD and are handled separately "
        "in the Castellar notebooks (flow_paper/notebooks/). "
        "Set LOCK='stim' for flow paper plots here."
    )

# ==============================================================================
# Auto-detect latest DATE_SOURCE
# ==============================================================================
DATE_SOURCE = show_latest_pkls(root, PAPER, PARADIGM, LOCK, MEASURE, BASELINE_CORR, cohort)
# Override if needed:
# DATE_SOURCE = "20260529"

In [ ]:
# ==============================================================================
# Derived labels and paths (auto — do not edit)
# ==============================================================================
_cfg = config_flow if PAPER == "flow" else config_load

CONDITIONS:      dict = _cfg.CONDITIONS
STYLES:          dict = _cfg.STYLES
BATCH_TOPO:      dict = _cfg.BATCH_TOPO      if hasattr(_cfg, "BATCH_TOPO")      else {}
BATCH_TOPO_RESP: dict = _cfg.BATCH_TOPO_RESP if hasattr(_cfg, "BATCH_TOPO_RESP") else {}

DEVIANT_LABEL = "target" if PARADIGM_REAL == "active" else "deviant"
label_csd     = "CSD" if CSD else "NOCSD"
DATE_STR      = datetime.today().strftime("%Y%m%d")

subfolder = "diff" if MEASURE == "diff" else "raw"

input_path = (
    root / "outputs" / "grand_averages" / PAPER / "_other_cohorts" / cohort / subfolder
    if cohort else
    root / "outputs" / "grand_averages" / PAPER / subfolder
)

root_img = (
    root / "outputs" / "img" / PAPER / "_other_cohorts" / cohort
    if cohort else
    root / "outputs" / "img" / PAPER
)
root_img.mkdir(parents=True, exist_ok=True)
(root_img / MEASURE).mkdir(parents=True, exist_ok=True)

# Event types and label map for axis labels
if LOCK == "resp":
    event_types    = ["response"]
    stim_label_map = {"response": "Response"}
elif MEASURE == "diff":
    event_types    = [f"{DEVIANT_LABEL}-standard", "novel-standard"]
    stim_label_map = {
        f"{DEVIANT_LABEL}-standard": f"{DEVIANT_LABEL.capitalize()}-Standard",
        "novel-standard":            "Novel-Standard",
    }
else:
    event_types    = ["standard", DEVIANT_LABEL, "novel"]
    stim_label_map = {
        "standard":    "Standard",
        DEVIANT_LABEL: DEVIANT_LABEL.capitalize(),
        "novel":       "Novel",
    }

print(f"Input  : {input_path}")
print(f"Output : {root_img}")
print(f"Events : {event_types}")

In [ ]:
# ==============================================================================
# Load pkl files and compute grand averages
# ==============================================================================

# N_PARTICIPANTS = {
#     "flow":  {"active": 36, "passive": 41},
#     "load":  {"active": 48, "passive": 47},
# }
# n = N_PARTICIPANTS[PAPER][PARADIGM]
n = len(get_participants(PAPER, PARADIGM))


def _build_fname(cond_label: str) -> str:
    base = f"{DATE_SOURCE}_{PARADIGM}_{n}p_cond-{cond_label}"
    if LOCK == "resp":
        return f"{base}_resp_{BASELINE_CORR}_{label_csd}_raw.pkl"
    elif MEASURE == "diff":
        return f"{base}_stim_diff.pkl"
    else:
        return f"{base}_stim_raw.pkl"


# Raw evokeds: {plot_label: {event_key: [Evoked, ...]}}
evokeds: dict[str, dict] = {}
for plot_label, cond_key in CONDITIONS.items():
    fname = _build_fname(cond_key)
    evokeds[plot_label] = load_pkl(input_path / fname)
    print(f"Loaded [{plot_label}] ← {fname}")

# Grand averages: {plot_label: {stim: Evoked}}
ga: dict[str, dict] = {label: {} for label in CONDITIONS}
for plot_label, cond_key in CONDITIONS.items():
    for stim in event_types:
        key = f"{cond_key}_{stim}"
        if key in evokeds[plot_label]:
            ga[plot_label][stim] = grand_average(evokeds[plot_label][key])
        else:
            print(f"Warning: key '{key}' not found for [{plot_label}]")

print("\nGrand averages computed.")
for label, stims in ga.items():
    print(f"  {label}: {list(stims.keys())}")


In [ ]:
# ==============================================================================
# plot_topomap_grid — core plot function
# ==============================================================================

def plot_topomap_grid(
    plot_windows,           # list of (stim, tmin_ms, tmax_ms)
    fname_suffix,
    manual_scale=None,
    scalings_override=None,
    fig_width=4,
    save=True,
    peak_mode=False,        # if True and LOCK=="resp": plot peak timepoint instead of window average
    show_peak_marker=False,
):
    col_labels = list(CONDITIONS.keys())
    n_cols = len(col_labels)
    n_rows = len(plot_windows)

    def peak_evoked(evoked, tmin_ms, tmax_ms):
        """Return a single-timepoint evoked at the peak latency within the window."""
        t_ms = evoked.times * 1000.0
        mask = (t_ms >= tmin_ms) & (t_ms <= tmax_ms)
        if not np.any(mask):
            raise ValueError(f"No samples inside {tmin_ms}-{tmax_ms} ms")
        # peak = max abs across all channels, within window
        windowed = evoked.data[:, mask]
        peak_col = np.argmax(np.abs(windowed).max(axis=0))
        peak_t_s = evoked.times[mask][peak_col]
        peak_t_ms = peak_t_s * 1000.0
        data_at_peak = evoked.data[:, mask][:, peak_col:peak_col+1]
        return mne.EvokedArray(
            data_at_peak, evoked.info.copy(),
            tmin=peak_t_s, nave=evoked.nave,
            comment=f"peak@{peak_t_ms:.0f}ms",
        ), peak_t_ms

    use_peak = peak_mode and LOCK == "resp"

    # Build row labels — for peak mode we'll patch them after computing peaks
    row_labels = [
        f"{stim_label_map.get(stim, stim.capitalize())}\n{a}–{b} ms"
        for stim, a, b in plot_windows
    ]
    peak_times_by_row: dict[int, float] = {}  # row → ms, filled below if use_peak

    def get_evoked_for_cell(label, stim, a, b):
        ev = ga[label][stim]
        if use_peak:
            ev_out, pt = peak_evoked(ev, a, b)
            return ev_out, pt
        else:
            return window_average_evoked(ev, a, b), None

    all_imgs = []
    for stim, a, b in plot_windows:
        for label in col_labels:
            if stim in ga[label]:
                ev, _ = get_evoked_for_cell(label, stim, a, b)
                all_imgs.append(get_topomap_image(ev))

    vmin = min(img.get_clim()[0] for img in all_imgs)
    vmax = max(img.get_clim()[1] for img in all_imgs)
    print(f"Auto scale: vmin={vmin:.4f}, vmax={vmax:.4f}")
    if manual_scale is not None:
        vmin, vmax = manual_scale
        print(f"Manual scale applied: vmin={vmin:.4f}, vmax={vmax:.4f}")

    last_ev, _ = get_evoked_for_cell(col_labels[0], plot_windows[-1][0], *plot_windows[-1][1:])
    ch_type = mne.channel_type(last_ev.info, 0)
    unit    = DEFAULTS["units"][ch_type]

    # ── figure ────────────────────────────────────────────────────────────────
    fig_height = n_rows * 2.5 + 1
    fig, axes  = plt.subplots(n_rows, n_cols, figsize=(fig_width * n_cols / 2, fig_height))
    if n_rows == 1:
        axes = axes[np.newaxis, :]

    images      = []
    topo_kwargs = dict(colorbar=False, show=False, time_format="")
    if scalings_override is not None:
        topo_kwargs["scalings"] = scalings_override

    for row, (stim, a, b) in enumerate(plot_windows):
        row_peak_times = []
        for col, label in enumerate(col_labels):
            if stim not in ga[label]:
                print(f"Warning: stim='{stim}' not in ga['{label}'] — skipping.")
                continue

            ev, peak_t = get_evoked_for_cell(label, stim, a, b)
            if peak_t is not None:
                row_peak_times.append(peak_t)

            data     = ev.data[:, 0]
            peak_idx = np.argmax(np.abs(data))
            peak_ch  = ev.info["ch_names"][peak_idx]
            peak_val = data[peak_idx]

            if show_peak_marker:
                mask              = np.zeros((len(ev.info["ch_names"]), len(ev.times)), dtype=bool)
                mask[peak_idx, :] = True
                ev.plot_topomap(
                    times=ev.times, axes=axes[row, col],
                    mask=mask,
                    mask_params=dict(
                        marker="o", markerfacecolor="none",
                        markeredgecolor="black", markersize=12, markeredgewidth=2,
                    ),
                    **topo_kwargs,
                )
            else:
                ev.plot_topomap(
                    times=ev.times, axes=axes[row, col],
                    **topo_kwargs,
                )

            img = axes[row, col].images[0]
            img.set_clim(vmin, vmax)
            images.append(img)

            if show_peak_marker:
                xlabel = f"{peak_ch}  ({peak_val * 1e6:.2f} µV)"
                if use_peak and peak_t is not None:
                    xlabel = f"{peak_t:.0f} ms | {xlabel}"
                axes[row, col].set_xlabel(xlabel, fontsize=9)

        # patch row label with peak time range if peak_mode
        if use_peak and row_peak_times:
            stim_label = stim_label_map.get(stim, stim.capitalize())
            pt_min, pt_max = min(row_peak_times), max(row_peak_times)
            if pt_min == pt_max:
                row_labels[row] = f"{stim_label}\npeak @ {pt_min:.0f} ms"
            else:
                row_labels[row] = f"{stim_label}\npeak {pt_min:.0f}, {pt_max:.0f} ms"

        axes[row, 0].set_ylabel(row_labels[row], fontsize=11, labelpad=12)

    for col, label in enumerate(col_labels):
        axes[0, col].set_title(label, fontsize=12)

    bottom_margin = max(0.04, 0.12 / n_rows)
    fig.subplots_adjust(bottom=bottom_margin, hspace=0.0)

    cbar_height = 0.08 / n_rows
    cbar_ax     = fig.add_axes([0.2, bottom_margin * 0.4, 0.6, cbar_height])
    cbar        = fig.colorbar(images[0], cax=cbar_ax, orientation="horizontal")
    cbar.set_label(f"Amplitude ({unit})")

    bc_label_title = f" ({BASELINE_CORR})" if LOCK == "resp" else ""
    bc_label_fname = f"_{BASELINE_CORR}"   if LOCK == "resp" else ""
    peak_marker_label = f"_peakmarker" if show_peak_marker == True else ""

    fig.suptitle(
        f"[{PARADIGM.capitalize()}]{bc_label_title}"
    )


    if save:
        n_p    = list(list(ga.values())[0].values())[0].nave
        mode   = "peak" if use_peak else "mean"
        fname  = (
            root_img
            / MEASURE
            / f"{PARADIGM}_{n_p}p_{PAPER}_{MEASURE}_topo"
              f"_{fname_suffix}_{mode}{bc_label_fname}{peak_marker_label}_{DATE_STR}.svg"
        )

        plt.savefig(fname, dpi=300, bbox_inches="tight")
        print(f"Saved → {fname}")
    plt.show()
    return fig, axes
print("plot_topomap_grid() ready.")


## Batch run

Runs all pre-configured topomap grids for the current `(PARADIGM, MEASURE)`.  
Edit configs in `config_flow.py` or `config_load.py` under `BATCH_TOPO`.


In [ ]:
batch_key     = (PARADIGM_REAL, MEASURE)
batch_topo    = BATCH_TOPO_RESP.get(batch_key, []) if LOCK == "resp" else BATCH_TOPO.get(batch_key, [])

if batch_topo:
    print(f"Running {len(batch_topo)} topo grids for ({PARADIGM_REAL}, {MEASURE}, {LOCK})...\n")
    for cfg in batch_topo:
        plot_topomap_grid(**cfg)
else:
    print(f"No BATCH_TOPO configs found for ({PARADIGM_REAL}, {MEASURE}, {LOCK}).")
    print("Add entries to BATCH_TOPO / BATCH_TOPO_RESP in config_flow.py / config_load.py,")
    print("or use the live cell below.")

## Live / custom plots

Use this cell for exploratory analysis or during meetings.


In [ ]:
# if MEASURE == "raw":
#     plot_topomap_grid(
#         plot_windows = [("deviant",350, 750)],
#         fname_suffix = "P3",
#         manual_scale = (-2, 2),   # e.g. (-3e-6, 3e-6) to override auto scale
#         save         = True,
#     )
